<a href="https://colab.research.google.com/github/Hukkana/whisper/blob/main/faster_whisper_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Google Colabの使い方

- **セルの実行:** `Ctrl + Enter`, `Shift + Enter`
- **セルの作成:** `Ctrl + M B`
- **セルの削除:** `Ctrl + M D`

In [1]:
!apt-get update -qq
!apt-get install -y build-essential libavformat-dev libavcodec-dev libavdevice-dev libavutil-dev libswscale-dev libx264-dev libx265-dev libvpx-dev python3-dev
!pip install --upgrade pip
!pip install faster-whisper==0.10.0 --force-reinstall

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
libx264-dev is already the newest version (2:0.163.3060+git5db6aa6-2build1).
libx265-dev is already the newest version (3.5-2).
libvpx-dev is already the newest version (1.11.0-2ubuntu2.5).
python3-dev is already the newest version (3.10.6-1~22.04.1).
libavcodec-dev is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
libavdevice-dev is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
libavformat-dev is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
libavutil-dev is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
libswscale-dev is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to

In [1]:
!pip show faster-whisper
import faster_whisper
print('faster_whisper successfully imported.')

Name: faster-whisper
Version: 0.10.0
Summary: Faster Whisper transcription with CTranslate2
Home-page: https://github.com/guillaumekln/faster-whisper
Author: Guillaume Klein
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: av, ctranslate2, huggingface-hub, onnxruntime, tokenizers
Required-by: 
faster_whisper successfully imported.


# Google Drive連携

In [2]:
# prompt: google driveにアクセスして動画ファイルを取得
import os
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
os.listdir("/content/drive/MyDrive/video/")

['daijiro.mp4']

In [4]:
path_video = "/content/drive/MyDrive/video/daijiro.mp4"

# Faster Whisper-v3ダウンロード


In [5]:
# srt file を作成するための関数
def create_srt_file(results, use_faster_whisper, file_name="transcribe"):
    data = []
    with open(f"{file_name}.srt", mode="w") as f:
        for index, _dict in enumerate(results):
            if use_faster_whisper:
              start_time = _dict.start # start
              end_time = _dict.end # end
              text = _dict.text # text
            else:
              start_time = _dict["start"]
              end_time = _dict["end"]
              text = _dict["text"]

            data.append({
            "index": index + 1,
            "start": start_time,
            "end": end_time,
            "text": text})

            # 時、分、秒、ミリ秒に分割
            s_h, s_m, s_s = int(start_time // 3600), int((start_time % 3600) // 60), int(start_time % 60)
            e_h, e_m, e_s = int(end_time // 3600), int((end_time % 3600) // 60), int(end_time % 60)

            # ミリ秒を計算
            s_ms = int((start_time - int(start_time)) * 1000)
            e_ms = int((end_time - int(end_time)) * 1000)

            f.write(f"{index+1}\n")
            f.write(f"{s_h:02}:{s_m:02}:{s_s:02},{s_ms:03} --> {e_h:02}:{e_m:02}:{e_s:02},{e_ms:03}\n")
            f.write(f"{text}\n\n")
    return data

In [6]:
import torch
import gc

def release_model_memory(model):
    """
    指定されたモデルをメモリから削除し、ガーベージコレクションとPyTorchのキャッシュメモリ解放を行う関数。

    Parameters:
    model (torch.nn.Module): メモリから解放するモデルオブジェクト。
    """
    # モデルを削除
    del model

    # ガーベージコレクションを実行してメモリを解放
    gc.collect()

    # PyTorchのキャッシュされたメモリを解放
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [7]:
from faster_whisper import WhisperModel

In [8]:
# model = WhisperModel("large-v3", device="cuda", compute_type="int8_float16")
model = WhisperModel("large-v3", device="cuda", compute_type="float16")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

vocabulary.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

In [9]:
%%time
segments, info = model.transcribe(path_video, beam_size=5)

CPU times: user 18.6 s, sys: 475 ms, total: 19.1 s
Wall time: 21.2 s


In [10]:
%%time
faster_whisper_data = create_srt_file(results=segments, use_faster_whisper=True, file_name="transcribe_faster_whisper")

CPU times: user 14min 2s, sys: 9.17 s, total: 14min 11s
Wall time: 12min 2s


# VAD filter

In [11]:
%%time
segments, info = model.transcribe(path_video, beam_size=5, vad_filter=True, vad_parameters=dict(min_silence_duration_ms=300))

CPU times: user 42.2 s, sys: 346 ms, total: 42.6 s
Wall time: 42.8 s


In [12]:
%%time
faster_whisper_vad_filter_data_300 = create_srt_file(results=segments, use_faster_whisper=True, file_name="transcribe_faster_whisper_vad_filter")

CPU times: user 12min 34s, sys: 8.06 s, total: 12min 42s
Wall time: 10min 54s


# whisper-v3

※ここからは検証用なので実行しなくても良いです

In [ ]:
# メモリ解放
release_model_memory(model)

In [ ]:
%%capture
!pip install openai-whisper==20231106

In [ ]:
# prompt: whisperをpythonで書く
import whisper

model = whisper.load_model("large-v3")

100%|█████████████████████████████████████| 2.88G/2.88G [01:07<00:00, 45.5MiB/s]


In [ ]:
%%time
result = model.transcribe(audio=path_video, fp16=True)

CPU times: user 1min 8s, sys: 365 ms, total: 1min 9s
Wall time: 1min 17s


In [ ]:
%%time
normal_whisper = create_srt_file(results=result["segments"], use_faster_whisper=False, file_name="transcribe_whisper")

CPU times: user 1.15 ms, sys: 17 µs, total: 1.17 ms
Wall time: 2.35 ms


# 結果の比較

In [ ]:
import pandas as pd

df_whisper = pd.DataFrame(normal_whisper)
df_faster_whisper = pd.DataFrame(faster_whisper_data)

In [ ]:
df_merge = pd.merge(df_faster_whisper[["index", "start", "end", "text"]],
                     df_whisper[["index", "start", "end", "text"]],
                     how="outer", on="index", suffixes=('_faster', '_normal'))

In [ ]:
df_merge.head(50)

,index,start_faster,end_faster,text_faster,start_normal,end_normal,text_normal
0,1,0.00,3.86,このアシスタントAPIを使うには最初にまずアシスタントというのを作ります,0.00,3.86,このアシスタントAPIを使うには最初にまずアシスタントというのを作ります
1,2,3.86,6.80,オリジナルのGPTが作れますよという感じですね,3.86,6.80,オリジナルのGPTが作れますよという感じですね
2,3,6.80,8.86,皆さんこんにちは、AI部長にゃんたです,6.80,8.86,皆さんこんにちは、AI部長にゃんたです
3,4,8.86,11.64,本日はオープンAIデブデイということで,8.86,11.64,本日はオープンAIデブデイということで
4,5,11.64,14.44,オープンAIから新しい機能がたくさん出てきたので,11.64,14.44,オープンAIから新しい機能がたくさん出てきたので
5,6,14.44,15.98,そちらについてまとめていきます,14.44,15.98,そちらについてまとめていきます
6,7,15.98,19.40,日本時間の11月7日の午前3時から,15.98,19.40,日本時間の11月7日の午前3時から
7,8,19.40,22.32,デベロッパーズコンファレンスというのが開催されてたんですが,19.40,22.32,Developers Conferenceというのが開催されていたんですが
8,9,22.32,23.60,皆さん見てみましたか?,22.32,23.60,皆さん見てみましたか?
9,10,24.00,26.20,午前3時からだったのでにゃんたはですね,24.00,26.20,午前3時からだったのでにゃんたはですね


# VAD Filterを使った場合

In [ ]:
df_faster_whisper_vadfilter_300 = pd.DataFrame(faster_whisper_vad_filter_data_300)

In [ ]:
df_merge = pd.merge(df_faster_whisper[["index", "start", "end", "text"]],
                     df_faster_whisper_vadfilter_300[["index", "start", "end", "text"]],
                     how="outer", on="index", suffixes=('_faster', '_vadfilter_300'))

In [ ]:
df_merge.head(50)

,index,start_faster,end_faster,text_faster,start_vadfilter_300,end_vadfilter_300,text_vadfilter_300
0,1,0.00,3.86,このアシスタントAPIを使うには最初にまずアシスタントというのを作ります,0.00,3.86,このアシスタントAPIを使うには最初にまずアシスタントというのを作ります
1,2,3.86,6.80,オリジナルのGPTが作れますよという感じですね,3.86,6.80,オリジナルのGPTが作れますよという感じですね
2,3,6.80,8.86,皆さんこんにちは、AI部長にゃんたです,6.80,8.86,皆さんこんにちは、AI部長にゃんたです
3,4,8.86,11.64,本日はオープンAIデブデイということで,8.86,11.64,本日はオープンAIデブデイということで
4,5,11.64,14.44,オープンAIから新しい機能がたくさん出てきたので,11.64,14.44,オープンAIから新しい機能がたくさん出てきたので
5,6,14.44,15.98,そちらについてまとめていきます,14.44,15.98,そちらについてまとめていきます
6,7,15.98,19.40,日本時間の11月7日の午前3時から,15.98,19.40,日本時間の11月7日の午前3時から
7,8,19.40,22.32,デベロッパーズコンファレンスというのが開催されてたんですが,19.40,22.32,デベロッパーズコンファレンスというのが開催されてたんですが
8,9,22.32,23.60,皆さん見てみましたか?,22.32,23.60,皆さん見てみましたか?
9,10,24.00,26.20,午前3時からだったのでにゃんたはですね,24.00,26.20,午前3時からだったのでにゃんたはですね
